# Test Visualization for U-Net Baseline on Crack500

This notebook loads the saved best checkpoint from the U-Net baseline and visualizes predictions on the Crack500 test set.

The main goals are:
- inspect qualitative prediction quality on fixed test samples
- compare ground-truth masks and predicted masks side by side
- create a reusable visualization workflow for later comparison with SegFormer-B2


In [ ]:
# ------------------------------------------------------------
# Code Cell 1: Imports and basic configuration
# What this cell should do:
# 1. import all required libraries
# 2. define device, checkpoint path, image size, and sample indices
# 3. keep these settings aligned with train.py and test.py
# ------------------------------------------------------------
import os
import sys

sys.path.insert(0, "..")  # Allow the notebook to import project files from the repo root.

import numpy as np
import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader

from dataset import CrackDataset
from loss import BCEDiceLoss
from metrics import f1_score, iou_score
from model import get_model

ROOT = "../CRACK500"
img_size = 360
batch_size = 16
threshold = 0.5
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
checkpoint_path = os.path.join("..", "checkpoints", "best_model.pth")

model_name = "Unet"
encoder_name = "resnet34"
encoder_weights = "imagenet"
in_channels = 3
classes = 1

sample_indices = [0, 5, 12, 25, 40]

plt.rcParams["figure.dpi"] = 120

print(f"Using device: {device}")
print(f"Checkpoint path: {checkpoint_path}")
print(f"Fixed sample indices: {sample_indices}")


## 1. Setup

This section defines the basic configuration for the notebook.

We keep the test-time settings consistent with the training and testing scripts:
- the same image size
- the same model architecture
- the same checkpoint path

We also choose a fixed list of test sample indices so that later visual comparisons remain stable across different models.


In [ ]:
# ------------------------------------------------------------
# Code Cell 2: Build the test dataset
# What this cell should do:
# 1. create the Crack500 test dataset
# 2. optionally print dataset length
# 3. confirm that the notebook is reading the correct split
# ------------------------------------------------------------
test_dataset = CrackDataset(ROOT, split="test", img_size=img_size)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=0)

print(f"Loaded test dataset with {len(test_dataset)} samples.")
raw_image, raw_mask = test_dataset.get_raw(0)
print(f"Sample 0 raw image shape: {raw_image.shape}")
print(f"Sample 0 raw mask shape: {raw_mask.shape}")


## 2. Load the Test Dataset

Here we load the `test` split of Crack500.

This notebook should only visualize the held-out test set, because the goal is to inspect the final baseline result rather than training behavior.


In [ ]:
# ------------------------------------------------------------
# Code Cell 3: Rebuild the model and load the saved checkpoint
# What this cell should do:
# 1. recreate the exact same U-Net model structure used in training
# 2. move the model to the selected device
# 3. load best_model.pth
# 4. switch the model to evaluation mode
# ------------------------------------------------------------
model = get_model(
    model_name=model_name,
    encoder_name=encoder_name,
    encoder_weights=encoder_weights,
    in_channels=in_channels,
    classes=classes,
)
model = model.to(device)
criterion = BCEDiceLoss()

if not os.path.exists(checkpoint_path):
    raise FileNotFoundError(
        f"Checkpoint not found at: {checkpoint_path}. "
        "Make sure training has saved best_model.pth before running this notebook."
    )

state_dict = torch.load(checkpoint_path, map_location=device, weights_only=True)
model.load_state_dict(state_dict)
model.eval()

print("Model rebuilt and checkpoint loaded successfully.")


## 3. Reload the Best Baseline Model

The checkpoint stores learned weights, not the model definition itself.

So we must:
1. rebuild the same model architecture
2. load the saved `state_dict`
3. switch the model to evaluation mode with `model.eval()`

This ensures that the predictions shown below come from the saved best validation checkpoint rather than from a random or last-epoch model state.


In [ ]:
# ------------------------------------------------------------
# Code Cell 4: Define a helper function for single-sample prediction
# What this cell should do:
# 1. take one dataset index
# 2. load the image and mask
# 3. run inference on that single sample
# 4. convert the prediction into a binary mask
# 5. return everything needed for plotting
# ------------------------------------------------------------
def make_overlay(image_np, mask_np, color=(255, 0, 0), alpha=0.45):
    overlay = image_np.copy().astype(np.float32)
    color_arr = np.array(color, dtype=np.float32)
    overlay[mask_np == 1] = overlay[mask_np == 1] * (1 - alpha) + color_arr * alpha
    return overlay.clip(0, 255).astype(np.uint8)


def predict_one(index, threshold=threshold):
    # Use get_raw for display so we keep the original RGB image and mask.
    raw_image, raw_mask = test_dataset.get_raw(index)

    # Use the transformed sample for model inference because the model expects
    # normalized tensors with the same preprocessing used during training.
    image_t, mask_t = test_dataset[index]
    image_t = image_t.unsqueeze(0).to(device)
    mask_t = mask_t.unsqueeze(0).unsqueeze(1).float().to(device)

    with torch.no_grad():
        logits = model(image_t)
        probs = torch.sigmoid(logits)
        pred_mask_t = (probs > threshold).float()

    # The model predicts on the resized tensor, but the raw image can keep
    # its original size. Resize the prediction back to raw-image size before
    # plotting or making overlays.
    raw_h, raw_w = raw_mask.shape
    pred_mask = torch.nn.functional.interpolate(
        pred_mask_t,
        size=(raw_h, raw_w),
        mode="nearest",
    ).squeeze().cpu().numpy().astype(np.uint8)
    prob_map = torch.nn.functional.interpolate(
        probs,
        size=(raw_h, raw_w),
        mode="bilinear",
        align_corners=False,
    ).squeeze().cpu().numpy()
    gt_mask = raw_mask.astype(np.uint8)

    sample_loss = criterion(logits, mask_t).item()
    sample_iou = iou_score(logits, mask_t, threshold=threshold)
    sample_f1 = f1_score(logits, mask_t, threshold=threshold)

    gt_overlay = make_overlay(raw_image, gt_mask, color=(0, 255, 0), alpha=0.35)
    pred_overlay = make_overlay(raw_image, pred_mask, color=(255, 0, 0), alpha=0.45)

    return {
        "index": index,
        "image": raw_image,
        "gt_mask": gt_mask,
        "pred_mask": pred_mask,
        "prob_map": prob_map,
        "gt_overlay": gt_overlay,
        "pred_overlay": pred_overlay,
        "loss": sample_loss,
        "iou": sample_iou,
        "f1": sample_f1,
    }


sample = predict_one(sample_indices[0])
print(
    f"Helper check on idx={sample['index']} | "
    f"loss={sample['loss']:.4f} | iou={sample['iou']:.4f} | f1={sample['f1']:.4f}"
)


## 4. Define a Prediction Helper

To keep the visualization clean, we define a small helper function for one sample at a time.

The helper should return:
- the original image
- the ground-truth mask
- the predicted mask
- optionally an overlay image

This makes the later plotting cells shorter and easier to read.


In [ ]:
# ------------------------------------------------------------
# Code Cell 5: Visualize fixed test samples
# What this cell should do:
# 1. loop over a fixed list of test indices
# 2. get image, ground truth, prediction, and overlay
# 3. plot them in a clean grid
# 4. keep the layout consistent for future model comparison
# ------------------------------------------------------------
num_rows = len(sample_indices)
fig, axes = plt.subplots(num_rows, 4, figsize=(14, 3 * num_rows))

if num_rows == 1:
    axes = np.expand_dims(axes, axis=0)

column_titles = ["Image", "Ground Truth", "Prediction", "Prediction Overlay"]
for col, title in enumerate(column_titles):
    axes[0, col].set_title(title, fontsize=11)

for row, idx in enumerate(sample_indices):
    sample = predict_one(idx)

    axes[row, 0].imshow(sample["image"])
    axes[row, 1].imshow(sample["gt_mask"], cmap="gray", vmin=0, vmax=1)
    axes[row, 2].imshow(sample["pred_mask"], cmap="gray", vmin=0, vmax=1)
    axes[row, 3].imshow(sample["pred_overlay"])

    axes[row, 0].set_ylabel(
        f"idx={idx}\nloss={sample['loss']:.3f}\nIoU={sample['iou']:.3f}\nF1={sample['f1']:.3f}",
        rotation=0,
        labelpad=45,
        va="center",
        fontsize=9,
    )

    for ax in axes[row]:
        ax.axis("off")

plt.tight_layout()
plt.show()


## 5. Visualize Fixed Test Samples

This section shows a small fixed set of test images.

For each sample, we want to display:
- the original image
- the ground-truth mask
- the predicted mask
- the prediction overlay on the original image

Using fixed sample indices is important because it allows direct visual comparison later when we run SegFormer-B2 on the same test cases.


In [ ]:
# ------------------------------------------------------------
# Code Cell 6: Optional qualitative analysis
# What this cell should do:
# 1. inspect where the model performs well
# 2. inspect where the model misses thin cracks or creates false positives
# 3. optionally print or annotate sample indices worth discussing
# ------------------------------------------------------------
test_loss = 0.0
test_iou = 0.0
test_f1 = 0.0

with torch.no_grad():
    for images, masks in test_loader:
        images = images.to(device)
        masks = masks.unsqueeze(1).float().to(device)

        outputs = model(images)
        loss = criterion(outputs, masks)

        batch_size_now = images.size(0)
        test_loss += loss.item() * batch_size_now
        test_iou += iou_score(outputs, masks, threshold=threshold) * batch_size_now
        test_f1 += f1_score(outputs, masks, threshold=threshold) * batch_size_now

test_loss /= len(test_dataset)
test_iou /= len(test_dataset)
test_f1 /= len(test_dataset)

print(f"Notebook test check | loss={test_loss:.4f} | iou={test_iou:.4f} | f1={test_f1:.4f}")
print()
print("Fixed-sample summary:")

for idx in sample_indices:
    sample = predict_one(idx)
    gt_pixels = int(sample["gt_mask"].sum())
    pred_pixels = int(sample["pred_mask"].sum())
    print(
        f"idx={idx:4d} | loss={sample['loss']:.4f} | "
        f"IoU={sample['iou']:.4f} | F1={sample['f1']:.4f} | "
        f"gt_pixels={gt_pixels} | pred_pixels={pred_pixels}"
    )


## 6. Qualitative Observations

While looking at the predicted masks, pay attention to:
- thin cracks that are partially missed
- false positives on textured background regions
- broken or disconnected crack segments
- cases where the model aligns well with the crack geometry

This section is useful for building intuition before moving on to stronger models such as SegFormer-B2.


In [ ]:
# ------------------------------------------------------------
# Code Cell 7: Optional failure-case collection
# What this cell should do:
# 1. select a few difficult or representative failure cases
# 2. visualize them separately
# 3. use them later for model comparison or discussion in the report
# ------------------------------------------------------------
num_cases_to_scan = min(100, len(test_dataset))
iou_records = []

for idx in range(num_cases_to_scan):
    sample = predict_one(idx)
    iou_records.append((sample["iou"], idx, sample["f1"]))

iou_records.sort(key=lambda x: x[0])
hard_cases = iou_records[:3]

print(f"Scanned the first {num_cases_to_scan} test samples.")
print("Lowest-IoU cases found:")
for rank, (sample_iou, idx, sample_f1) in enumerate(hard_cases, start=1):
    print(f"{rank}. idx={idx} | IoU={sample_iou:.4f} | F1={sample_f1:.4f}")

fig, axes = plt.subplots(len(hard_cases), 4, figsize=(14, 3 * len(hard_cases)))

if len(hard_cases) == 1:
    axes = np.expand_dims(axes, axis=0)

column_titles = ["Image", "Ground Truth", "Prediction", "Prediction Overlay"]
for col, title in enumerate(column_titles):
    axes[0, col].set_title(title, fontsize=11)

for row, (sample_iou, idx, sample_f1) in enumerate(hard_cases):
    sample = predict_one(idx)

    axes[row, 0].imshow(sample["image"])
    axes[row, 1].imshow(sample["gt_mask"], cmap="gray", vmin=0, vmax=1)
    axes[row, 2].imshow(sample["pred_mask"], cmap="gray", vmin=0, vmax=1)
    axes[row, 3].imshow(sample["pred_overlay"])

    axes[row, 0].set_ylabel(
        f"idx={idx}\nIoU={sample_iou:.3f}\nF1={sample_f1:.3f}",
        rotation=0,
        labelpad=40,
        va="center",
        fontsize=9,
    )

    for ax in axes[row]:
        ax.axis("off")

plt.tight_layout()
plt.show()


## 7. Failure Cases for Later Comparison

The current low-IoU cases do not all reflect the same type of failure, so they should be interpreted carefully.

Observed patterns in this U-Net baseline:
- Some top-ranked failure cases have `pred_pixels = 0`.
- In the first two zero-prediction cases, even human inspection does not reveal a clearly visible crack in the source image; the labeled area appears only as a slightly darker region than the surrounding pavement.
- These examples are better described as extremely low-contrast or visually ambiguous cases, rather than simple misses on an obvious crack.
- In other failure cases, the model usually follows the general crack location and direction, but the predicted mask is thinner than the ground truth and is broken into disconnected segments.
- This suggests that the baseline struggles more with weak visibility, crack width recovery, and continuity than with coarse crack localization.

These cases are still especially valuable because they can later be reused to compare:
- U-Net vs SegFormer-B2
- in-domain vs cross-domain behavior
- clean test images vs perturbed test images

Keeping the same difficult examples across experiments will make the later qualitative discussion much stronger and more honest.


## 8. Summary

This notebook provides a qualitative view of the current U-Net baseline on the Crack500 test set.

It serves two purposes:
- verify that the saved best checkpoint produces sensible crack masks
- establish a consistent visualization format for future experiments

The same notebook structure can later be reused for SegFormer-B2 and for cross-domain UAV evaluation.
